# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id), at a single snapshot in time — not a time series. There's no date column in this starter dataset; instead, "age" and "freshness" are captured as pre-computed windows (content_age_days, days_since_last_update, impressions_90d = trailing 90 days). This means I can't observe true before/after outcomes here — every feature and the label are drawn from the same snapshot window, which is a limitation I address in Section 4.

In [1]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
print("Unique content_id values:", df["content_id"].nunique())
print("Duplicate content_id rows:", df.shape[0] - df["content_id"].nunique())

30000 rows, 44 columns
Unique content_id values: 30000
Duplicate content_id rows: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: impressions_90d, search_volume, competition, cpc, word_count, char_count, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier.

Label: trend_direction (used to build is_churned).

Context (join keys, not features): content_id, client_id.

Excluded: trend_pct — this is excluded because it's the exact percentage the trend_direction label is computed from, so including it would leak the answer directly into the model (confirmed as leakage in Notebook 02).

In [2]:
feature_cols = ["impressions_90d","search_volume","competition","cpc","word_count","char_count",
                "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
                "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
                "competition_level","content_type","main_intent","age_tier","freshness_tier",
                "word_count_tier","impression_tier","position_tier"]
label_col = "trend_direction"
context_cols = ["content_id", "client_id"]
excluded_cols = ["trend_pct"]

print("Feature columns present:", [c for c in feature_cols if c in df.columns])
print("Missing from data:", [c for c in feature_cols if c not in df.columns])

Feature columns present: ['impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Missing from data: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Checking the grain claim (one row per page), missing values in key fields, and confirming there's genuinely no date/time column — just pre-aggregated windows.

In [3]:
print("Grain check — duplicate content_id count:", df.shape[0] - df["content_id"].nunique())
print()
print("Missing values in key fields:")
print(df[["impressions_90d","content_age_days","days_since_last_update","trend_direction"]].isnull().sum())
print()
date_like_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
print("Date/time-like columns found:", date_like_cols)

Grain check — duplicate content_id count: 0

Missing values in key fields:
impressions_90d           0
content_age_days          0
days_since_last_update    0
trend_direction           0
dtype: int64

Date/time-like columns found: ['days_since_last_update']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me about true future outcomes — it's a single snapshot, so trend_direction reflects a recent window, not something that happens strictly after a decision point. It also can't reveal per-client differences directly since client_id is anonymized and I have no client metadata beyond the id. Any conclusion I draw is directional and tied to this one snapshot; it won't necessarily generalize to a different time period without re-verification against the full warehouse release later in the internship.

In [4]:
print("Clients represented:", df["client_id"].nunique())
print("Snapshot limitation: single point in time, no forward-looking validation possible with this file alone.")

Clients represented: 32
Snapshot limitation: single point in time, no forward-looking validation possible with this file alone.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.